In [1]:
from math import pi
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

In [2]:
def qft_dagger(qc, n):
    """Inverse QFT on the first n qubits."""
    for qubit in range(n // 2):
        qc.swap(qubit, n - qubit - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-pi / float(2**(j - m)), m, j)
        qc.h(j)
        
def qpe_main(qc, angle, target):
    """Controlled-U^(2^k) for each counting qubit k, U = phase(angle)."""
    repetitions = 1
    for control in range(target):
        for _ in range(repetitions):
            qc.cp(angle, control, target)
        repetitions *= 2

In [ ]:
# ---- (a) S gate: theta = 1/4 with 3 counting qubits ----
angle = pi / 2                                # S = diag(1, i) -> phase pi/2
qpe = QuantumCircuit(4, 3)
for i in range(3): qpe.h(i)
qpe.x(3)                                      # target |1>
qpe.barrier()
qpe_main(qpe, angle, 3)
qpe.barrier()
qft_dagger(qpe, 3)
qpe.barrier()
for i in range(3): qpe.measure(i, i)
backend = Aer.get_backend('qasm_simulator')
counts = backend.run(transpile(qpe, backend), shots=1000).result().get_counts()
print(counts)                                 # {'010': 1000}  -> 2 -> theta = 2/8 = 1/4
plot_histogram(counts)

In [4]:
# ---- (b) theta = 1/5 with 8 counting qubits ----
angle = 2 * pi / 5
t = 8
qpe1 = QuantumCircuit(t + 1, t)
for i in range(t): qpe1.h(i)
qpe1.x(t)
qpe1.barrier()
qpe_main(qpe1, angle, t)
qpe1.barrier()
qft_dagger(qpe1, t)
qpe1.barrier()
for i in range(t): qpe1.measure(i, i)
counts1 = backend.run(transpile(qpe1, backend), shots=1000).result().get_counts()
best = max(counts1, key=counts1.get)
m = int(best, 2)                              # counts string is already binary, MSB on the left
print(best, m, m / 2**t)  

00110011 51 0.19921875
